# Mutator Demo — Generate Jailbreak Variants

This notebook:
- Loads the unified dataset from `../data/unified/unified_dataset.csv`
- Samples seed jailbreak prompts
- Runs `src/mutator.py` strategies to generate adversarial variants

Outputs are printed inline for inspection.


## 0. Install dependencies (Colab only)


In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q -r ../requirements.txt
    print('Colab: installed requirements.txt')
else:
    print('Local/Jupyter: using your existing environment')


## 1. Imports


In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

# Make ../src importable
SRC_DIR = (Path.cwd().parent / 'src').resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from mutator import JailbreakMutator

print('SRC_DIR:', SRC_DIR)


## 2. Load unified dataset


In [ ]:
unified_path = Path('..') / 'data' / 'unified' / 'unified_dataset.csv'
if not unified_path.exists():
    raise FileNotFoundError(
        f"Missing {unified_path}. Run notebooks/setup.ipynb first to generate it."
    )

df = pd.read_csv(unified_path)
print('Rows:', len(df))
print('Columns:', list(df.columns))
print('Label distribution:')
print(df['jailbreak'].value_counts())


## 3. Generate variants


In [ ]:
# Pick a few jailbreak seeds
seed_df = df[df['jailbreak'] == 1].sample(min(5, (df['jailbreak'] == 1).sum()), random_state=42)
seeds = seed_df['prompt'].tolist()

mutator = JailbreakMutator(
    strategies=['wordnet', 'roleplay', 'structural'],
    combine=False,
    strategy_kwargs={
        'wordnet': {'swap_rate': 0.25},
        'structural': {'shuffle_sentences': True, 'insert_filler': True, 'homoglyph_noise': 0.02},
    }
)

for i, seed in enumerate(seeds, 1):
    print('\n' + '='*90)
    print(f'SEED {i}:')
    print(seed)
    print('-'*90)
    variants = mutator.mutate(seed, n=6)
    for j, v in enumerate(variants, 1):
        print(f'  Variant {j}: {v}')
